# Component 1 — B3 Shared-Private and B4 Hierarchy-Guided Transfer Models

In [ ]:

from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import precision_recall_fscore_support, f1_score, accuracy_score, hamming_loss
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from IPython.display import display

CURRENT = Path.cwd().resolve()
if CURRENT.name == "notebooks":
    PROJECT_ROOT = CURRENT.parent
elif (CURRENT / "data").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "data").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise RuntimeError(f"Could not locate component_1 from {CURRENT}")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mabsa_multilabel_reviews.csv"
TAXONOMY_PATH = PROJECT_ROOT / "taxonomy" / "aspect_taxonomy_v0_2.yaml"
RESULTS_DIR = PROJECT_ROOT / "results" / "transfer"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "aspect_b4"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 192
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP_NORM = 1.0
PROJECTION_DIM = 256
DROPOUT = 0.20
SMOKE_TEST = False
SAVE_MODEL_WEIGHTS = False
SAVE_FINAL_ARTIFACTS = True
ASPECT_MODEL_VERSION = "b4_hgss_v1"
TAXONOMY_VERSION = "0.2"
RUN_OPTIONAL_B4_NO_GATE_ABLATION = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)


In [ ]:

for path in [DATA_PATH, TAXONOMY_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

data = pd.read_csv(DATA_PATH)
data["canonical_aspects"] = data["canonical_aspects"].apply(json.loads)

with open(TAXONOMY_PATH, "r", encoding="utf-8") as f:
    taxonomy = yaml.safe_load(f)

COMMON = taxonomy["groups"]["COMMON"]
PHONE_SPECIFIC = taxonomy["groups"]["PHONE_SPECIFIC"]
LAPTOP_SPECIFIC = taxonomy["groups"]["LAPTOP_SPECIFIC"]

GLOBAL_LABELS = COMMON + PHONE_SPECIFIC + LAPTOP_SPECIFIC
LABEL_TO_ID = {label: i for i, label in enumerate(GLOBAL_LABELS)}

COMMON_IDX = list(range(0, len(COMMON)))
PHONE_IDX = list(range(len(COMMON), len(COMMON) + len(PHONE_SPECIFIC)))
LAPTOP_IDX = list(
    range(
        len(COMMON) + len(PHONE_SPECIFIC),
        len(GLOBAL_LABELS),
    )
)

DOMAIN_TO_ID = {"phone": 0, "laptop": 1}

train_df = data[data["split"].eq("train")].reset_index(drop=True)
dev_df = data[data["split"].eq("dev")].reset_index(drop=True)
test_df = data[data["split"].eq("test")].reset_index(drop=True)

if SMOKE_TEST:
    train_df = train_df.head(256)
    dev_df = dev_df.head(128)
    test_df = test_df.head(128)

print("Global labels:", len(GLOBAL_LABELS))
display(
    data.groupby(["domain", "split"])
    .size()
    .rename("count")
    .reset_index()
)


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TransferDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

        self.enc = tokenizer(
            self.df["text"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

        self.labels = np.zeros(
            (len(self.df), len(GLOBAL_LABELS)),
            dtype=np.float32,
        )

        self.valid_masks = np.zeros_like(self.labels)

        for i, row in self.df.iterrows():
            for aspect in row["canonical_aspects"]:
                self.labels[i, LABEL_TO_ID[aspect]] = 1.0

            self.valid_masks[i, COMMON_IDX] = 1.0

            if row["domain"] == "phone":
                self.valid_masks[i, PHONE_IDX] = 1.0
            else:
                self.valid_masks[i, LAPTOP_IDX] = 1.0

        self.domain_ids = (
            self.df["domain"]
            .map(DOMAIN_TO_ID)
            .astype(int)
            .to_numpy()
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item = {
            k: torch.tensor(v[idx])
            for k, v in self.enc.items()
        }
        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.float32,
        )
        item["valid_mask"] = torch.tensor(
            self.valid_masks[idx],
            dtype=torch.float32,
        )
        item["domain_id"] = torch.tensor(
            self.domain_ids[idx],
            dtype=torch.long,
        )
        return item


def make_loader(df, shuffle=False):
    return DataLoader(
        TransferDataset(df),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
    )


train_loader = make_loader(train_df, shuffle=True)
dev_loader = make_loader(dev_df)
test_loader = make_loader(test_df)


In [ ]:

class B3GenericSharedPrivate(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden = self.encoder.config.hidden_size

        self.shared_proj = nn.Sequential(
            nn.Linear(hidden, PROJECTION_DIM),
            nn.Tanh(),
            nn.Dropout(DROPOUT),
        )

        self.phone_private = nn.Sequential(
            nn.Linear(hidden, PROJECTION_DIM),
            nn.Tanh(),
            nn.Dropout(DROPOUT),
        )

        self.laptop_private = nn.Sequential(
            nn.Linear(hidden, PROJECTION_DIM),
            nn.Tanh(),
            nn.Dropout(DROPOUT),
        )

        self.classifier = nn.Linear(
            PROJECTION_DIM * 2,
            len(GLOBAL_LABELS),
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        domain_id,
    ):
        pooled = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state[:, 0, :]

        shared = self.shared_proj(pooled)
        phone_private = self.phone_private(pooled)
        laptop_private = self.laptop_private(pooled)

        private = torch.where(
            domain_id.unsqueeze(1).eq(0),
            phone_private,
            laptop_private,
        )

        fused = torch.cat([shared, private], dim=1)

        return self.classifier(fused)


class B4HierarchyGuided(nn.Module):
    def __init__(self, use_gate=True):
        super().__init__()

        self.use_gate = use_gate

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden = self.encoder.config.hidden_size

        self.shared_proj = nn.Sequential(
            nn.Linear(hidden, PROJECTION_DIM),
            nn.Tanh(),
            nn.Dropout(DROPOUT),
        )

        self.phone_private = nn.Sequential(
            nn.Linear(hidden, PROJECTION_DIM),
            nn.Tanh(),
            nn.Dropout(DROPOUT),
        )

        self.laptop_private = nn.Sequential(
            nn.Linear(hidden, PROJECTION_DIM),
            nn.Tanh(),
            nn.Dropout(DROPOUT),
        )

        self.gate = nn.Linear(
            PROJECTION_DIM * 2,
            PROJECTION_DIM,
        )

        self.common_head = nn.Linear(
            PROJECTION_DIM,
            len(COMMON),
        )
        self.phone_head = nn.Linear(
            PROJECTION_DIM,
            len(PHONE_SPECIFIC),
        )
        self.laptop_head = nn.Linear(
            PROJECTION_DIM,
            len(LAPTOP_SPECIFIC),
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        domain_id,
    ):
        pooled = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state[:, 0, :]

        shared = self.shared_proj(pooled)
        phone_private = self.phone_private(pooled)
        laptop_private = self.laptop_private(pooled)

        private = torch.where(
            domain_id.unsqueeze(1).eq(0),
            phone_private,
            laptop_private,
        )

        if self.use_gate:
            gate = torch.sigmoid(
                self.gate(
                    torch.cat([shared, private], dim=1)
                )
            )
            specific_repr = (
                gate * shared
                + (1.0 - gate) * private
            )
        else:
            specific_repr = private

        common_logits = self.common_head(shared)
        phone_logits = self.phone_head(specific_repr)
        laptop_logits = self.laptop_head(specific_repr)

        phone_valid = domain_id.unsqueeze(1).eq(0)
        laptop_valid = domain_id.unsqueeze(1).eq(1)

        phone_logits = torch.where(
            phone_valid,
            phone_logits,
            torch.full_like(phone_logits, -20.0),
        )

        laptop_logits = torch.where(
            laptop_valid,
            laptop_logits,
            torch.full_like(laptop_logits, -20.0),
        )

        return torch.cat(
            [
                common_logits,
                phone_logits,
                laptop_logits,
            ],
            dim=1,
        )


In [ ]:

def masked_bce_loss(
    logits,
    labels,
    valid_mask,
    use_mask,
):
    raw = torch.nn.functional.binary_cross_entropy_with_logits(
        logits,
        labels,
        reduction="none",
    )

    if not use_mask:
        return raw.mean()

    masked = raw * valid_mask

    return masked.sum() / valid_mask.sum().clamp_min(1.0)


def collect_probs(model, loader):
    model.eval()

    probs_list = []
    labels_list = []
    masks_list = []
    domains_list = []

    with torch.no_grad():
        for batch in loader:
            inputs = {
                "input_ids": batch["input_ids"].to(DEVICE),
                "attention_mask": batch["attention_mask"].to(DEVICE),
                "domain_id": batch["domain_id"].to(DEVICE),
            }

            logits = model(**inputs)
            probs = torch.sigmoid(logits).cpu().numpy()

            probs_list.append(probs)
            labels_list.append(batch["labels"].numpy())
            masks_list.append(batch["valid_mask"].numpy())
            domains_list.append(batch["domain_id"].numpy())

    return (
        np.vstack(probs_list),
        np.vstack(labels_list),
        np.vstack(masks_list),
        np.concatenate(domains_list),
    )


def tune_threshold(
    y_true,
    probs,
    valid_mask=None,
):
    rows = []

    for threshold in np.arange(0.10, 0.91, 0.05):
        pred = (probs >= threshold).astype(int)

        if valid_mask is None:
            score = f1_score(
                y_true,
                pred,
                average="micro",
                zero_division=0,
            )
        else:
            mask = valid_mask.astype(bool)
            score = f1_score(
                y_true[mask],
                pred[mask],
                average="binary",
                zero_division=0,
            )

        rows.append({
            "threshold": round(float(threshold), 2),
            "micro_f1": float(score),
        })

    table = pd.DataFrame(rows)

    best = table.sort_values(
        ["micro_f1", "threshold"],
        ascending=[False, True],
    ).iloc[0]

    return float(best["threshold"]), table


def compute_metrics(
    y_true,
    probs,
    threshold,
    valid_mask=None,
):
    pred = (probs >= threshold).astype(int)

    if valid_mask is not None:
        mask = valid_mask.astype(bool)
        y_flat = y_true[mask]
        p_flat = pred[mask]

        precision, recall, f1, _ = (
            precision_recall_fscore_support(
                y_flat,
                p_flat,
                average="binary",
                zero_division=0,
            )
        )

        return {
            "precision_micro": float(precision),
            "recall_micro": float(recall),
            "micro_f1": float(f1),
            "macro_f1": np.nan,
            "exact_match": np.nan,
            "hamming_loss": float(
                np.mean(y_flat != p_flat)
            ),
        }, pred

    p_micro, r_micro, f_micro, _ = (
        precision_recall_fscore_support(
            y_true,
            pred,
            average="micro",
            zero_division=0,
        )
    )

    _, _, f_macro, _ = (
        precision_recall_fscore_support(
            y_true,
            pred,
            average="macro",
            zero_division=0,
        )
    )

    return {
        "precision_micro": float(p_micro),
        "recall_micro": float(r_micro),
        "micro_f1": float(f_micro),
        "macro_f1": float(f_macro),
        "exact_match": float(
            accuracy_score(y_true, pred)
        ),
        "hamming_loss": float(
            hamming_loss(y_true, pred)
        ),
    }, pred


In [ ]:

def train_transfer_model(
    model_name,
    model,
    use_hierarchy_mask,
):
    model = model.to(DEVICE)

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = EPOCHS * len(train_loader)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            total_steps * WARMUP_RATIO
        ),
        num_training_steps=total_steps,
    )

    history = []
    start = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for batch in tqdm(
            train_loader,
            desc=f"{model_name} epoch {epoch}/{EPOCHS}",
        ):
            labels = batch["labels"].to(DEVICE)
            valid_mask = batch["valid_mask"].to(DEVICE)

            inputs = {
                "input_ids": batch["input_ids"].to(DEVICE),
                "attention_mask": batch["attention_mask"].to(DEVICE),
                "domain_id": batch["domain_id"].to(DEVICE),
            }

            optimizer.zero_grad()

            logits = model(**inputs)

            loss = masked_bce_loss(
                logits,
                labels,
                valid_mask,
                use_mask=use_hierarchy_mask,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP_NORM,
            )

            optimizer.step()
            scheduler.step()

            losses.append(loss.item())

        model.eval()
        dev_losses = []

        with torch.no_grad():
            for batch in dev_loader:
                labels = batch["labels"].to(DEVICE)
                valid_mask = batch["valid_mask"].to(DEVICE)

                inputs = {
                    "input_ids": batch["input_ids"].to(DEVICE),
                    "attention_mask": batch["attention_mask"].to(DEVICE),
                    "domain_id": batch["domain_id"].to(DEVICE),
                }

                logits = model(**inputs)

                loss = masked_bce_loss(
                    logits,
                    labels,
                    valid_mask,
                    use_mask=use_hierarchy_mask,
                )

                dev_losses.append(loss.item())

        row = {
            "model": model_name,
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "dev_loss": float(np.mean(dev_losses)),
        }

        history.append(row)
        print(row)

    dev_probs, dev_true, dev_masks, _ = (
        collect_probs(
            model,
            dev_loader,
        )
    )

    threshold, threshold_table = tune_threshold(
        dev_true,
        dev_probs,
        valid_mask=(
            dev_masks
            if use_hierarchy_mask
            else None
        ),
    )

    (
        test_probs,
        test_true,
        test_masks,
        test_domains,
    ) = collect_probs(
        model,
        test_loader,
    )

    metrics, test_pred = compute_metrics(
        test_true,
        test_probs,
        threshold,
        valid_mask=(
            test_masks
            if use_hierarchy_mask
            else None
        ),
    )

    if SAVE_MODEL_WEIGHTS:
        torch.save(
            model.state_dict(),
            RESULTS_DIR / f"{model_name}_state_dict.pt",
        )

    return {
        "model": model,
        "metrics": metrics,
        "threshold": threshold,
        "history": pd.DataFrame(history),
        "threshold_table": threshold_table.assign(
            model=model_name
        ),
        "test_probs": test_probs,
        "test_true": test_true,
        "test_pred": test_pred,
        "test_masks": test_masks,
        "test_domains": test_domains,
        "training_seconds": time.time() - start,
        "use_hierarchy_mask": use_hierarchy_mask,
    }


In [ ]:

b3 = train_transfer_model(
    "b3_shared_private",
    B3GenericSharedPrivate(),
    use_hierarchy_mask=False,
)

b4 = train_transfer_model(
    "b4_hgss",
    B4HierarchyGuided(use_gate=True),
    use_hierarchy_mask=True,
)

results = {
    "b3": b3,
    "b4": b4,
}

metric_rows = []

for model_key, result in results.items():
    metric_rows.append({
        "model": model_key,
        "segment": "overall",
        **result["metrics"],
        "threshold": result["threshold"],
        "training_seconds": result["training_seconds"],
    })

    for domain, domain_id in DOMAIN_TO_ID.items():
        idx = (
            result["test_domains"] == domain_id
        )

        domain_metrics, _ = compute_metrics(
            result["test_true"][idx],
            result["test_probs"][idx],
            result["threshold"],
            valid_mask=(
                result["test_masks"][idx]
                if result["use_hierarchy_mask"]
                else None
            ),
        )

        metric_rows.append({
            "model": model_key,
            "segment": domain,
            **domain_metrics,
            "threshold": result["threshold"],
            "training_seconds": result["training_seconds"],
        })

metrics_df = pd.DataFrame(metric_rows)

display(metrics_df)


In [ ]:

def group_metric_rows(
    model_key,
    result,
):
    rows = []

    group_specs = [
        (
            "COMMON",
            COMMON_IDX,
            None,
        ),
        (
            "PHONE_SPECIFIC",
            PHONE_IDX,
            0,
        ),
        (
            "LAPTOP_SPECIFIC",
            LAPTOP_IDX,
            1,
        ),
    ]

    for (
        group_name,
        indices,
        required_domain,
    ) in group_specs:
        sample_mask = np.ones(
            len(test_df),
            dtype=bool,
        )

        if required_domain is not None:
            sample_mask = (
                result["test_domains"]
                == required_domain
            )

        y_true = (
            result["test_true"][sample_mask]
            [:, indices]
        )
        probs = (
            result["test_probs"][sample_mask]
            [:, indices]
        )

        group_metrics, _ = compute_metrics(
            y_true,
            probs,
            result["threshold"],
            valid_mask=None,
        )

        rows.append({
            "model": model_key,
            "semantic_group": group_name,
            **group_metrics,
        })

    return rows


group_metrics_df = pd.DataFrame(
    group_metric_rows("b3", b3)
    + group_metric_rows("b4", b4)
)

prediction_parts = []

for model_key, result in results.items():
    rows = []

    for i, row in test_df.reset_index(
        drop=True
    ).iterrows():
        true_aspects = [
            GLOBAL_LABELS[j]
            for j in range(
                len(GLOBAL_LABELS)
            )
            if result["test_true"][i, j] == 1
        ]

        predicted_aspects = [
            GLOBAL_LABELS[j]
            for j in range(
                len(GLOBAL_LABELS)
            )
            if result["test_pred"][i, j] == 1
        ]

        rows.append({
            "model": model_key,
            "domain": row["domain"],
            "review_id": row["review_id"],
            "text": row["text"],
            "true_aspects": json.dumps(
                true_aspects,
                ensure_ascii=False,
            ),
            "predicted_aspects": json.dumps(
                predicted_aspects,
                ensure_ascii=False,
            ),
            "threshold": result["threshold"],
        })

    prediction_parts.append(
        pd.DataFrame(rows)
    )

predictions_df = pd.concat(
    prediction_parts,
    ignore_index=True,
)

history_df = pd.concat(
    [
        b3["history"],
        b4["history"],
    ],
    ignore_index=True,
)

display(group_metrics_df)


In [ ]:
if SAVE_FINAL_ARTIFACTS:
    b4_state_path = ARTIFACT_DIR / "b4_hgss_state_dict.pt"
    b4_config_path = ARTIFACT_DIR / "b4_inference_config.json"
    tokenizer_dir = ARTIFACT_DIR / "tokenizer"

    torch.save(b4["model"].state_dict(), b4_state_path)
    tokenizer.save_pretrained(tokenizer_dir)

    semantic_group_by_aspect = {
        **{aspect: "COMMON" for aspect in COMMON},
        **{aspect: "PHONE_SPECIFIC" for aspect in PHONE_SPECIFIC},
        **{aspect: "LAPTOP_SPECIFIC" for aspect in LAPTOP_SPECIFIC}
    }

    b4_inference_config = {
        "model_version": ASPECT_MODEL_VERSION,
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "projection_dim": PROJECTION_DIM,
        "dropout": DROPOUT,
        "threshold": float(b4["threshold"]),
        "taxonomy_version": TAXONOMY_VERSION,
        "common_labels": COMMON,
        "phone_specific_labels": PHONE_SPECIFIC,
        "laptop_specific_labels": LAPTOP_SPECIFIC,
        "global_labels": GLOBAL_LABELS,
        "semantic_group_by_aspect": semantic_group_by_aspect,
        "domain_to_id": DOMAIN_TO_ID,
        "aspect_probability_note": (
            "Raw sigmoid probability used for aspect selection; not claimed to be calibrated."
        )
    }

    with open(b4_config_path, "w", encoding="utf-8") as f:
        json.dump(b4_inference_config, f, ensure_ascii=False, indent=2)

    print("Saved final aspect artifacts:")
    print(b4_state_path)
    print(b4_config_path)
    print(tokenizer_dir)

In [ ]:

ablation_df = pd.DataFrame()

if RUN_OPTIONAL_B4_NO_GATE_ABLATION:
    b4_no_gate = train_transfer_model(
        "b4_no_gate",
        B4HierarchyGuided(
            use_gate=False
        ),
        use_hierarchy_mask=True,
    )

    ablation_df = pd.DataFrame([
        {
            "variant": "b4_full",
            **b4["metrics"],
        },
        {
            "variant": "b4_no_gate",
            **b4_no_gate["metrics"],
        },
    ])

    ablation_df.to_csv(
        RESULTS_DIR / "b4_ablation_metrics.csv",
        index=False,
    )

    del b4_no_gate["model"]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics_df.to_csv(
    RESULTS_DIR / "transfer_model_metrics.csv",
    index=False,
)

group_metrics_df.to_csv(
    RESULTS_DIR / "transfer_group_metrics.csv",
    index=False,
)

history_df.to_csv(
    RESULTS_DIR / "transfer_training_history.csv",
    index=False,
)

predictions_df.to_csv(
    RESULTS_DIR / "transfer_test_predictions.csv",
    index=False,
)

comparison_df = (
    metrics_df[
        metrics_df["segment"].isin(
            ["phone", "laptop"]
        )
    ]
    .pivot(
        index="segment",
        columns="model",
        values=["micro_f1", "macro_f1"],
    )
)

comparison_df.to_csv(
    RESULTS_DIR / "b3_b4_comparison.csv"
)

with open(
    RESULTS_DIR / "transfer_run_config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "model_name": MODEL_NAME,
            "seed": SEED,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "projection_dim": PROJECTION_DIM,
            "dropout": DROPOUT,
            "common_labels": COMMON,
            "phone_specific_labels": PHONE_SPECIFIC,
            "laptop_specific_labels": LAPTOP_SPECIFIC,
            "b3": (
                "generic shared/private representation "
                "without semantic hierarchy"
            ),
            "b4": (
                "hierarchy-guided common/specific routing "
                "with selective fusion"
            ),
            "smoke_test": SMOKE_TEST,
            "save_model_weights": SAVE_MODEL_WEIGHTS,
            "device": str(DEVICE),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

for result in [b3, b4]:
    del result["model"]

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:

required_outputs = [
    RESULTS_DIR / "transfer_model_metrics.csv",
    RESULTS_DIR / "transfer_group_metrics.csv",
    RESULTS_DIR / "transfer_training_history.csv",
    RESULTS_DIR / "transfer_test_predictions.csv",
    RESULTS_DIR / "b3_b4_comparison.csv",
    RESULTS_DIR / "transfer_run_config.json",
]

artifact_outputs = [
    ARTIFACT_DIR / "b4_hgss_state_dict.pt",
    ARTIFACT_DIR / "b4_inference_config.json",
    ARTIFACT_DIR / "tokenizer",
]

checks = {
    "b3_metrics_created":
        "b3" in set(metrics_df["model"]),

    "b4_metrics_created":
        "b4" in set(metrics_df["model"]),

    "all_groups_evaluated":
        set(
            group_metrics_df["semantic_group"]
        )
        == {
            "COMMON",
            "PHONE_SPECIFIC",
            "LAPTOP_SPECIFIC",
        },

    "micro_f1_valid":
        metrics_df[
            "micro_f1"
        ].dropna().between(0, 1).all(),

    "outputs_saved":
        all(
            path.exists()
            for path in required_outputs
        ),

    "final_artifacts_saved": (
        all(path.exists() for path in artifact_outputs)
        if SAVE_FINAL_ARTIFACTS else True
    ),
}

validation_df = pd.DataFrame(
    [
        {
            "check": key,
            "passed": bool(value),
        }
        for key, value in checks.items()
    ]
)

display(validation_df)

if not validation_df["passed"].all():
    failed = validation_df.loc[
        ~validation_df["passed"],
        "check",
    ].tolist()

    raise ValueError(
        f"B3/B4 VALIDATION FAILED: {failed}"
    )

print("C1 B3 + B4 TRANSFER MODELS: PASSED")
